# NFN — Neural Fractal Network · Lightning.ai Training

**Free tier:** 22 GPU-hours/month · GPU: T4 (16 GB)

### Setup
1. Go to [lightning.ai](https://lightning.ai) → Sign up free
2. New Studio → upload this notebook OR paste from GitHub
3. Enable GPU: top-right `CPU` → select `T4`
4. Run all cells

**Advantage over Colab:** Lightning Studios **persist between sessions** — no need to re-download or re-install.

**Preset:** `medium_wikipedia` — 85M params, ~12h, checkpoints persist automatically.

In [ ]:
# ── Cell 1: Install (only needed once — Lightning persists the env) ───────────
import os, subprocess

if not os.path.exists('/teamspace/studios/this_studio/FNN'):
    subprocess.run(['git', 'clone', 'https://github.com/AFKmoney/FNN.git',
                    '/teamspace/studios/this_studio/FNN'], check=True)
else:
    subprocess.run(['git', '-C', '/teamspace/studios/this_studio/FNN', 'pull'])

os.chdir('/teamspace/studios/this_studio/FNN')
os.system('pip install -e . -q && pip install tqdm -q')
print('✓ Ready — files persist in /teamspace/studios/this_studio/FNN')

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — enable GPU in top-right"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else '')
print(f'PyTorch {torch.__version__}')

In [ ]:
# ── Cell 3: Download dataset (cached on disk — only runs once) ────────────────
import os, sys
os.chdir('/teamspace/studios/this_studio/FNN')
sys.path.insert(0, '.')

DATA_FILE = 'data/train_wiki.txt'
if not os.path.exists(DATA_FILE):
    from datasets.downloader import DatasetDownloader
    dl   = DatasetDownloader('data')
    text = dl.get('wikipedia-en-simple', max_chars=80_000_000)
    os.makedirs('data', exist_ok=True)
    open(DATA_FILE, 'w').write(text)
    print(f'✓ Downloaded {len(text):,} chars')
else:
    size = os.path.getsize(DATA_FILE)
    print(f'✓ Dataset already cached ({size/1e6:.1f} MB)')

In [ ]:
# ── Cell 4: Train (runs up to 12h per session, auto-resumes) ──────────────────
import os
os.chdir('/teamspace/studios/this_studio/FNN')

resume = '--resume checkpoints/agi_nfn_latest.pt' \
         if os.path.exists('checkpoints/agi_nfn_latest.pt') else ''
print('▶ Resuming...' if resume else '▶ Starting fresh...')

# medium config — fits T4 16GB at batch=6, seq=512
!python train_agi.py \
    --text data/train_wiki.txt \
    --config medium \
    --epochs 3 \
    --batch 6 \
    --seq-len 512 \
    --lr 2e-4 \
    --fp16 \
    --sample-every 500 \
    --save-every 500 \
    {resume}

In [ ]:
# ── Cell 5: Generate text ─────────────────────────────────────────────────────
import torch, sys, os
os.chdir('/teamspace/studios/this_studio/FNN')
sys.path.insert(0, '.')

from nfn.agi_model import build_agi_model
from nfn.tokenizer import NFNTokenizer

tok   = NFNTokenizer()
ckpt  = torch.load('checkpoints/agi_nfn_final.pt', map_location='cpu')
model = build_agi_model(vocab_size=tok.vocab_size, d_model=512, n_blocks=8)
model.load_state_dict(ckpt['model_state'])
model.eval()

for prompt in ['The universe is', 'Scientists have discovered', 'Language models']:
    ids = tok.encode(prompt, add_bos=True)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=100, temperature=0.8)
    print(f'\n>>> {prompt}')
    print(tok.decode(out[0].tolist()))